In [ ]:
# Import necessary libraries for data processing
from itertools import product

import pandas as pd
import os
import json
import importlib

In [ ]:
schema_lookup = dict()

In [ ]:
# Placeholder code for schema to be read from files and store in schema_lookup
schema_lookup = {
    # Your KGs
    "kwg": {
        "axiom": "schemas/kwg/axiom.txt",
        "nen": "schemas/kwg/nen.txt",
        "ttl": "schemas/kwg/schema.ttl"
    },
    "kwg_lite": {
        "axiom": "schemas/kwg_lite/axiom.txt",
        "nen": "schemas/kwg_lite/nen.txt",
        "ttl": "schemas/kwg_lite/schema.ttl"
    },
    "enslaved": {
        "axiom": "schemas/enslaved/axiom.txt",
        "nen": "schemas/enslaved/nen.txt",
        "ttl": "schemas/enslaved/schema.ttl"
    },
    "enslaved_wiki": {
        "axiom": "schemas/enslaved_wiki/axiom.txt",
        "nen": "schemas/enslaved_wiki/nen.txt",
        "ttl": "schemas/enslaved_wiki/schema.ttl"
    },
    "core_scholar_rich": {
        "axiom": "schemas/core_scholar_rich/axiom.txt",
        "nen": "schemas/core_scholar_rich/nen.txt",
        "ttl": "schemas/core_scholar_rich/schema.ttl"
    },
    "core_scholar_shallow": {
        "axiom": "schemas/core_scholar_shallow/axiom.txt",
        "nen": "schemas/core_scholar_shallow/nen.txt",
        "ttl": "schemas/core_scholar_shallow/schema.ttl"
    },

    "gbo": {
        "axiom": "schemas/gbo/axiom.txt",
        "nen": "schemas/gbo/nen.txt",
        "ttl": "schemas/gbo/schema.ttl"
    },
    "gmo": {
        "axiom": "schemas/gmo/axiom.txt",
        "nen": "schemas/gmo/nen.txt",
        "ttl": "schemas/gmo/schema.ttl"
    },
    "currkg": {
        "axiom": "schemas/currkg/axiom.txt",
        "nen": "schemas/currkg/nen.txt",
        "ttl": "schemas/currkg/schema.ttl"
    }
}

In [ ]:
def load_file_to_string(file_path):
  """Loads the content of a file into a string.

  Args:
    file_path: The path to the file.

  Returns:
    The content of the file as a string.
  """
  with open(file_path, "r", encoding="utf-8") as file:
    return file.read()

In [ ]:
def fill_prompt_template(template_text, values_dict):
    for key, value in values_dict.items():
        template_text = template_text.replace(f"{{{key}}}", value)
    return template_text

In [ ]:
def get_CQ_list(ids):
    """Given a list of IDs, returns the non-empty competency questions from their files."""
    cq_list = []
    for id in ids:
        cq_blob = load_file_to_string(f"cqs/{id}.txt")
        questions = [line.strip() for line in cq_blob.splitlines() if line.strip()]
        cq_list.extend(questions)
    return cq_list

In [ ]:
def build_kg_variation_context(ids, representations, schema_lookup=schema_lookup):
    blocks = []

    for kg_id in ids:
        if kg_id not in schema_lookup:
            print(f"KG-{kg_id} not found in schema lookup.")
            continue

        rep_blocks = []

        for rep in representations:
            rep_key = rep.lower()

            if rep_key not in schema_lookup[kg_id]:
                print(f"Representation '{rep}' not found for KG-{kg_id}.")
                continue

            file_name = schema_lookup[kg_id][rep_key]
            rep_content = load_file_to_string(file_name)
            rep_blocks.append(rep_content)

        joined_rep_blocks = "\n\n".join(rep_blocks)
        blocks.append(f"KG-{kg_id} schema context:\n{joined_rep_blocks}")

    return "\n\n".join(blocks)

In [ ]:
def load_prompt_module(task, prompt_type):
    module_name = f"prompts.{task}_{prompt_type}"
    prompt_module = importlib.import_module(module_name)
    prompt_module = importlib.reload(prompt_module)

    if not hasattr(prompt_module, "SYSTEM_PROMPT"):
        raise ValueError(f"{module_name} does not define SYSTEM_PROMPT")
    if not hasattr(prompt_module, "USER_PROMPT_TEMPLATE"):
        raise ValueError(f"{module_name} does not define USER_PROMPT_TEMPLATE")

    return prompt_module.SYSTEM_PROMPT, prompt_module.USER_PROMPT_TEMPLATE

In [ ]:
experiment_setups = [
    # KG selection experiments
    # single schema
    # 0-shot
    {"kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich", "consolidated_kgs"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_sel"},
    {"kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich", "consolidated_kgs"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_sel"},
    {"kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich", "consolidated_kgs"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_sel"},

    # cot
    {"kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich", "consolidated_kgs"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_sel"},
    {"kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich", "consolidated_kgs"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_sel"},
    {"kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich", "consolidated_kgs"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_sel"},
    
    # multiple schemas
    # nen, 0-shot
    {"cq_ids": ["2bridge_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_currkg_kwg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_kwg"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_currkg"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_currkg"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_enslaved"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_enslaved"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_gmo"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_gmo"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},

    {"cq_ids": ["3bridge_enslaved_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_enslaved_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_gmo_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_gmo_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_enslaved_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_gmo_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    
    # axiom, 0-shot
    {"cq_ids": ["2bridge_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_currkg_kwg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_kwg"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_currkg"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_currkg"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_enslaved"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_enslaved"],"representations":["axiom"],"temperatures":[0.0],"prompt_types":["0_shot"],"task":"kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_gmo"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_gmo"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},

    {"cq_ids": ["3bridge_enslaved_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_enslaved_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_gmo_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_gmo_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_enslaved_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_gmo_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},

    # ttl, 0-shot
    {"cq_ids": ["2bridge_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_currkg_kwg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_kwg"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_currkg"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_currkg"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_enslaved"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_enslaved"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_gmo"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_gmo"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},

    {"cq_ids": ["3bridge_enslaved_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_enslaved_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_gmo_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_gmo_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_enslaved_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_gmo_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["0_shot"], "task": "kg_mul_sel"},

    # nen, cot
    {"cq_ids": ["2bridge_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_currkg_kwg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_kwg"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_currkg"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_currkg"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_enslaved"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_enslaved"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_gmo"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_gmo"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},

    {"cq_ids": ["3bridge_enslaved_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_enslaved_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_gmo_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_gmo_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_currkg_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_enslaved_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_gmo_core_scholar"], "representations": ["nen"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},

    # axiom, cot
    {"cq_ids": ["2bridge_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_currkg_kwg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_kwg"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_currkg"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_currkg"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_enslaved"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_enslaved"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_gmo"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_gmo"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},

    {"cq_ids": ["3bridge_enslaved_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_enslaved_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_gmo_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_gmo_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_currkg_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_enslaved_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_gmo_core_scholar"], "representations": ["axiom"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},

    # ttl, cot
    {"cq_ids": ["2bridge_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_currkg_kwg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_currkg_kwg"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_enslaved_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_enslaved_currkg"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_gmo_currkg"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_gmo_currkg"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_enslaved"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_enslaved"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["2bridge_kwg_gmo"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["2bridge_kwg_gmo"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},

    {"cq_ids": ["3bridge_enslaved_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_enslaved_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_gmo_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_gmo_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_currkg_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_currkg_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_enslaved_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_enslaved_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"},
    {"cq_ids": ["3bridge_kwg_gmo_core_scholar"], "kg_ids": ["kwg", "enslaved", "currkg", "gmo", "core_scholar_rich"], "prompt_asset_ids": ["3bridge_kwg_gmo_core_scholar"], "representations": ["ttl"], "temperatures": [0.0], "prompt_types": ["cot"], "task": "kg_mul_sel"}
]   

In [ ]:
import random
PROMPT_ASSET_DIR = "prompt_assets"

def load_prompt_asset(task, kg_id, allow_default=True):
    asset_path = os.path.join(PROMPT_ASSET_DIR, task, f"{kg_id}.json")
    if not os.path.exists(asset_path):
        if not allow_default:
            raise FileNotFoundError(f"Prompt asset not found: {asset_path}")
        asset_path = os.path.join(PROMPT_ASSET_DIR, task, "default.json")

    if os.environ.get("DEBUG_PROMPT_ASSETS"):
        print(f"Loaded prompt asset: task={task} id={kg_id} path={asset_path}")
    with open(asset_path, "r", encoding="utf-8") as file:
        return json.load(file)

def format_kg_selection_examples(examples):
    formatted_examples = []
    for index, example in enumerate(examples, start=1):
        formatted_examples.append(
            f"Example {index}:\n"
            f"Competency Question: {example['cq']}\n"
            f"Answer: {example['answer']}"
        )

    return "\n\n".join(formatted_examples)

def format_sparql_examples(examples):
    formatted_examples = []
    for index, example in enumerate(examples, start=1):
        formatted_examples.append(
            f"Example {index}:\n"
            f"Competency Question: {example['cq']}\n"
            f"Answer: {example['query']}"
        )

    return "\n\n".join(formatted_examples)

def build_reasoning_example(example, answer_key, format_reasoning_list=False):
    reasoning = example["reasoning"]
    if format_reasoning_list and isinstance(reasoning, list):
        reasoning = "\n".join(f"- {step}" for step in reasoning)

    return (
        "Example 1:\n"
        f"Competency Question: {example['cq']}\n"
        f"Reasoning: {reasoning}\n"
        f"Answer: {example[answer_key]}"
    )

def load_prompt_support(task, kg_ids, prompt_asset_ids=None):
    asset_ids = prompt_asset_ids or kg_ids
    allow_default_asset = prompt_asset_ids is None
    assets = [load_prompt_asset(task, asset_id, allow_default=allow_default_asset) for asset_id in asset_ids]
    default_asset = load_prompt_asset(task, "default")
    random_asset_index = random.randint(0, len(assets) - 1) if assets else None

    reasoning_steps = assets[random_asset_index].get("reasoning_steps") or default_asset.get("reasoning_steps", [])

    if task in ("kg_sel", "kg_mul_sel"):
        few_shot_examples = []
        for asset in assets:
            few_shot_examples.extend(asset.get("few_shot_examples", []))

        if not few_shot_examples:
            few_shot_examples = default_asset.get("few_shot_examples", [])
        
        random.shuffle(few_shot_examples)  # Randomize the order of few-shot examples
        few_shot_examples = few_shot_examples[:5]  # Limit to 5 examples for prompt

        reasoning_asset = assets[random_asset_index] if assets else default_asset
        reasoning_example = build_reasoning_example(
            reasoning_asset.get("reasoning_example", default_asset.get("reasoning_example", {})),
            "answer",
            format_reasoning_list=(task == "kg_mul_sel"),
        )

        return {
            "few_shot_examples": format_kg_selection_examples(few_shot_examples),
            "reasoning_steps": "\n".join(reasoning_steps),
            "reasoning_example": reasoning_example,
        }

    if task == "sparql":
        asset = assets[0] if assets else default_asset
        few_shot_examples = asset.get("few_shot_examples") or default_asset.get("few_shot_examples", [])
        reasoning_example = build_reasoning_example(
            asset.get("reasoning_example", default_asset.get("reasoning_example", {})),
            "query",
        )

        return {
            "few_shot_examples": format_sparql_examples(few_shot_examples),
            "reasoning_steps": "\n".join(reasoning_steps),
            "reasoning_example": reasoning_example,
        }

    raise ValueError(f"Unsupported task: {task}")

# Local Models

In [ ]:
# Placeholder code for local model with ollama interface - replace with actual implementation when available
# from ollama_interface import chat_with_model

def interact_with_agent(model_name, system_message, user_prompt, temperature):
    """Interacts with a model agent using system and user prompts."""
    
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_prompt}
    ]

    # response = chat_with_model(
    #     model_name=model_name,
    #     messages=messages,
    #     options = {
    #         "temperature": temperature
    #     }
    # )
    # return response,response["message"]["content"]
    return 'response','content'

In [ ]:
models_to_run = []

In [ ]:
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

for setup in experiment_setups:
    cq_ids = setup.get("cq_ids", setup["kg_ids"])
    kg_ids = setup["kg_ids"]
    representations = setup["representations"]
    temperatures = setup["temperatures"]
    prompt_types = setup["prompt_types"]
    task = setup["task"]
    prompt_asset_ids = setup.get("prompt_asset_ids")
    if prompt_asset_ids is None and task == "kg_mul_sel":
        prompt_asset_ids = cq_ids

    print("-" * 100)

    # Load CQs once per setup
    cqs = get_CQ_list(cq_ids)

    # Build schema context once per setup
    schema_context = build_kg_variation_context(ids=kg_ids, representations=representations, schema_lookup=schema_lookup)
    prompt_support = load_prompt_support(task, kg_ids, prompt_asset_ids=prompt_asset_ids)

    for prompt_type, temperature, model in product(prompt_types, temperatures, models_to_run):
        print("-" * 100)
        print(f"Running setup task={task} | prompt_type={prompt_type} | temp={temperature} | model={model}")

        try:
            system_prompt, user_prompt_template = load_prompt_module(task, prompt_type)
            cq_model_results = []

            for cq in cqs:
                if task in ("kg_sel", "kg_mul_sel"):
                    input_data = {
                        "Insert_CQ_here": cq,
                        "Insert_schemas_here": schema_context,
                    }

                    if prompt_type == "few_shot":
                        input_data["Insert_examples_here"] = prompt_support["few_shot_examples"]
                    elif prompt_type == "cot":
                        input_data["Insert_reasoning_here"] = prompt_support["reasoning_steps"]
                        input_data["Insert_reasoning_example_here"] = prompt_support["reasoning_example"]

                elif task == "sparql":
                    input_data = {
                        "Insert_CQ_here": cq,
                        "Insert_schema_here": schema_context,
                    }

                    if prompt_type == "few_shot":
                        input_data["Insert_examples_here"] = prompt_support["few_shot_examples"]
                    elif prompt_type == "cot":
                        input_data["Insert_reasoning_here"] = prompt_support["reasoning_steps"]
                        input_data["Insert_reasoning_example_here"] = prompt_support["reasoning_example"]
                else:
                    raise ValueError(f"Unsupported task: {task}")

                filled_prompt = fill_prompt_template(user_prompt_template, input_data)
                

                model_analysis_raw, model_analysis_result = interact_with_agent(
                    model_name=model,
                    system_message=system_prompt,
                    user_prompt=filled_prompt,
                    temperature=temperature
                )

                cq_model_results.append({
                    "Task": task,
                    "Model": model,
                    "Prompt_Type": prompt_type,
                    "Temperature": temperature,
                    "KG_IDs": ", ".join(kg_ids),
                    "Representations": ", ".join(representations),
                    "CQ": cq,
                    "User_Prompt": filled_prompt,
                    "Analysis_Result": model_analysis_result,
                    "Analysis_Raw": str(model_analysis_raw),
                })

            df = pd.DataFrame(cq_model_results)


            file_name = (
                f"{task}"
                f"__kg-{'-'.join(kg_ids)}"
                f"__rep-{'-'.join(representations)}"
                f"__{prompt_type}"
                f"__{temperature}"
                f"__model-{model.replace(":","-")}.xlsx"
            )
            

            excel_file_path = os.path.join(results_dir, file_name)
            # print(filled_prompt, file=open(f'{excel_file_path}.txt', 'w'))
            df.to_excel(excel_file_path, index=False)

            print(f"Saved: {excel_file_path}")

        except Exception as e:
            print( f"task={task} | prompt_type={prompt_type} | temp={temperature} | model={model} | error={e}")

# Gemini

In [ ]:
model = "gemini-2.5-pro"

## JSON Batch Creator

In [ ]:
json_requests = []

for setup in experiment_setups:
    cq_ids = setup.get("cq_ids", setup["kg_ids"])
    kg_ids = setup["kg_ids"]
    representations = setup["representations"]
    temperatures = setup["temperatures"]
    prompt_types = setup["prompt_types"]
    task = setup["task"]
    prompt_asset_ids = setup.get("prompt_asset_ids")
    if prompt_asset_ids is None and task == "kg_mul_sel":
        prompt_asset_ids = cq_ids

    print("-" * 100)

    # Load CQs once per setup
    cqs = get_CQ_list(cq_ids)

    # Build schema context once per setup
    schema_context = build_kg_variation_context(ids=kg_ids, representations=representations, schema_lookup=schema_lookup)
    prompt_support = load_prompt_support(task, kg_ids, prompt_asset_ids=prompt_asset_ids)

    for prompt_type, temperature in product(prompt_types, temperatures):
        print("-" * 100)
        print(f"Running setup task={task} | representations={representations} | prompt_type={prompt_type} | temp={temperature} | model={model}")

        try:
            system_prompt, user_prompt_template = load_prompt_module(task, prompt_type)
            cq_model_results = []

            for cq in cqs:
                if task in ("kg_sel", "kg_mul_sel"):
                    input_data = {
                        "Insert_CQ_here": cq,
                        "Insert_schemas_here": schema_context,
                    }

                    if prompt_type == "few_shot":
                        input_data["Insert_examples_here"] = prompt_support["few_shot_examples"]
                    elif prompt_type == "cot":
                        input_data["Insert_reasoning_here"] = prompt_support["reasoning_steps"]
                        input_data["Insert_reasoning_example_here"] = prompt_support["reasoning_example"]

                elif task == "sparql":
                    input_data = {
                        "Insert_CQ_here": cq,
                        "Insert_schema_here": schema_context,
                    }

                    if prompt_type == "few_shot":
                        input_data["Insert_examples_here"] = prompt_support["few_shot_examples"]
                    elif prompt_type == "cot":
                        input_data["Insert_reasoning_here"] = prompt_support["reasoning_steps"]
                        input_data["Insert_reasoning_example_here"] = prompt_support["reasoning_example"]
                else:
                    raise ValueError(f"Unsupported task: {task}")

                filled_prompt = fill_prompt_template(user_prompt_template, input_data)

                json_requests.append({
                    "key": f"{task}-{'-'.join(representations)}-{'-'.join(kg_ids)}-{prompt_type}-temp{temperature}-{cq}",
                    "request": {
                        "system_instruction": {"parts": [{"text": system_prompt}]},
                        "contents": [
                            {
                                "role": "user",
                                "parts": [{"text": filled_prompt}]
                            }
                        ]
                    }
                })

        except Exception as e:
            print( f"task={task} | representations={representations} | prompt_type={prompt_type} | temp={temperature} | model={model} | error={e}")

In [ ]:
import json

# Use 'a' to append or 'w' to overwrite
with open('json_requests_sparql.jsonl', 'w', encoding='utf-8') as f:
    for request_data in json_requests:
        # json.dumps converts the entire dictionary to a string on ONE line
        # it automatically escapes internal newlines (\n) correctly
        line = json.dumps(request_data)
        f.write(line + '\n')

## Auth Google for Gemini

In [ ]:
!gcloud auth list
# !pip3 install google-cloud-storage

In [ ]:
from google import genai
import google.auth
from google.genai import types

# gemini api batch
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

credentials, projectId = google.auth.default()
client = genai.Client(
    vertexai=True,
    project=projectId,
    location="us-central1",
    credentials=credentials
)

## Queue Batch Job to Gemini

In [ ]:
# Cell 1: Upload local file to GCS and define paths
from google.cloud import storage

# 1. Define your local file and bucket details
local_file_path = "json_requests_sparql.jsonl"
bucket_name = "sparql_nen_axiom_ttl"
destination_blob_name = "json_requests_sparql.jsonl" # How it will be named in the bucket

# 2. Initialize the GCS client and upload the file
storage_client = storage.Client()
bucket = storage_client.bucket(bucket_name)
blob = bucket.blob(destination_blob_name)

print(f"Uploading {local_file_path} to gs://{bucket_name}/{destination_blob_name}...")
blob.upload_from_filename(local_file_path)
print("Upload complete!")

# 3. Define the URIs needed for the Vertex AI Batch Job in the next cell
gcs_input_uri = f"gs://{bucket_name}/{destination_blob_name}"
gcs_output_prefix = f"gs://{bucket_name}/batch_results/"

print(f"\nInput URI ready: {gcs_input_uri}")
print(f"Output will be saved to: {gcs_output_prefix}")

In [ ]:
from google import genai
from google.genai import types

# Assumes `uploaded_file` is the file object from the previous step
file_batch_job = client.batches.create(
    model=model,
    src=gcs_input_uri,
    config=types.CreateBatchJobConfig(
        dest=gcs_output_prefix,
        display_name="json-requests-batch-job"
    )
)

print(f"Created batch job: {file_batch_job.name}")

## Job Status check and result retrieval

In [ ]:
# Cell 3: Check job status and fetch results from GCS
from google.cloud import storage

# Fetch the current state of the job
job_name = file_batch_job.name
batch_job = client.batches.get(name=job_name)

print(f"Current Job State: {batch_job.state.name}")

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    print(f"Job completed! Results are stored in GCS at: {gcs_output_prefix}")
    
    print("Downloading result file content from GCS...")
    
    # Initialize the standard GCS client to read the output
    storage_client = storage.Client()
    # bucket_name = "sparql_nen_axiom_ttl"
    bucket = storage_client.bucket(bucket_name)
    
    # Vertex AI outputs results into the prefix folder you specified
    output_folder_prefix = "batch_results/"
    blobs = bucket.list_blobs(prefix=output_folder_prefix)
    
    # Iterate through the output files and print them
    for blob in blobs:
        # Vertex AI typically writes the output as .jsonl files
        if blob.name.endswith('.jsonl'):
            print(f"\n--- Contents of {blob.name} ---")
            # Download and decode the text directly
            file_content = blob.download_as_text()
            print(file_content)

elif batch_job.state.name == 'JOB_STATE_FAILED':
    print("Job failed.")
    if batch_job.error:
         print(f"Error: {batch_job.error}")
else:
    print("Job is still running or pending. Run this cell again later.")

In [ ]:
import json
from google.cloud import storage

# 1. Initialize the GCS client
storage_client = storage.Client()
bucket_name = "kgsel_cot_temp0_nen_axiom_ttl"
bucket = storage_client.bucket(bucket_name)

# 2. CHOOSE YOUR FILE NAME HERE
# Make sure to include the folder path if it's inside one!
# Example: "batch_results/predictions_00001.jsonl"
# chosen_file_name = "batch_results/prediction-model-2026-04-28T22:05:39.288457Z/predictions.jsonl" # kgsel_cot_temp0_nen_axiom_ttl 100%
# chosen_file_name = "batch_results/prediction-model-2026-04-29T15:44:55.204628Z/predictions.jsonl" # kgsel_0shot_temp0_nen_axiom_ttl 100%
# chosen_file_name = "batch_results/prediction-model-2026-05-01T16:45:58.024383Z/predictions.jsonl" # kgsel_0shot_temp0_nen_axiom_ttl 100% (with false sets)
chosen_file_name = "batch_results/prediction-model-2026-05-01T17:10:25.615457Z/predictions.jsonl" # kgsel_cot_temp0_nen_axiom_ttl 100% (with false sets)

# 3. Fetch that specific file directly (No iterator loop needed)
blob = bucket.blob(chosen_file_name)

print(f"\n--- Responses from {chosen_file_name} ---")

try:
    # Download the raw text from the chosen file
    file_content = blob.download_as_text()
    
    # 4. Iterate through the lines of the file
    for line in file_content.strip().split('\n'):
        if not line:
            continue # Skip empty lines
        
        try:
            data = json.loads(line)
            
            # Check if there's a valid response with candidates
            if "candidates" in data.get("response", {}):
                # Dig down to the actual text string
                generated_text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
                print(generated_text)
                print("-" * 40) # Separator between responses
                
            else:
                # If a specific prompt failed
                print(f"Failed Request Status: {data.get('status')}")
                print("-" * 40)
                
        except json.JSONDecodeError:
            print("Error parsing this line.")

except Exception as e:
    print(f"Error accessing the file. Make sure the file name is exactly correct. Details: {e}")

In [ ]:
import json

# 1. Define your local file name
# local_file_name = "batch_results_prediction-model-2026-04-28T22_05_39.288457Z_predictions.jsonl" # kgsel_cot_temp0_nen_axiom_ttl 100%
# local_file_name = "batch_results_prediction-model-2026-04-29T15_44_55.204628Z_predictions.jsonl" # kgsel_0shot_temp0_nen_axiom_ttl 100%
# local_file_name = "batch_results_prediction-model-2026-05-01T16_45_58.024383Z_predictions.jsonl" # kgsel_0shot_temp0_nen_axiom_ttl 100% (with false sets)
local_file_name = "batch_results_prediction-model-2026-05-01T17_10_25.615457Z_predictions.jsonl" # kgsel_cot_temp0_nen_axiom_ttl 100% (with false sets)


print(f"\n--- Responses from {local_file_name} ---")

try:
    # 2. Open the local file for reading
    with open(local_file_name, 'r', encoding='utf-8') as f:
        
        # 3. Iterate through the file line by line
        for line in f:
            line = line.strip()
            if not line:
                continue # Skip empty lines
            
            try:
                data = json.loads(line)
                
                # Check if there's a valid response with candidates
                if "candidates" in data.get("response", {}):
                    # Dig down to the actual text string
                    generated_text = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
                    
                    # Optional: Grab the 'key' if you want to know which prompt generated this answer
                    request_key = data.get("key", "Unknown")
                    print(f"Key: {request_key}")
                    
                    print(generated_text)
                    print("-" * 40) # Separator between responses
                    
                else:
                    # If a specific prompt failed
                    print(f"Failed Request Status: {data.get('status')}")
                    print("-" * 40)
                    
            except json.JSONDecodeError:
                print("Error parsing this line.")

except FileNotFoundError:
    print(f"Error: The file '{local_file_name}' was not found.")
    print("Make sure the file is in the exact same folder where you are running this notebook/script.")
except Exception as e:
    print(f"An unexpected error occurred. Details: {e}")

In [ ]:
import os
import glob
import json
import re
from collections import defaultdict

# --- Configuration ---
cqs_directory = "cqs" # The directory containing your .txt files
# local_file_name = "batch_results_prediction-model-2026-04-28T22_05_39.288457Z_predictions.jsonl" # kgsel_cot_temp0_nen_axiom_ttl 100%
# local_file_name = "batch_results_prediction-model-2026-04-29T15_44_55.204628Z_predictions.jsonl" # kgsel_0shot_temp0_nen_axiom_ttl 100%
# local_file_name = "batch_results_prediction-model-2026-05-01T16_45_58.024383Z_predictions.jsonl" # kgsel_0shot_temp0_nen_axiom_ttl 100% (with false sets)
local_file_name = "batch_results_prediction-model-2026-05-01T17_10_25.615457Z_predictions.jsonl" # kgsel_cot_temp0_nen_axiom_ttl 100% (with false sets)


print(f"\n--- Responses from {local_file_name} ---")

# TOGGLE THIS: Set to True to print mismatch details at the end, False to hide them
SHOW_NON_MATCH_DETAILS = True 

# --- 1. Build a mapping of Question -> Set of Expected Labels ---
question_to_expected_labels = defaultdict(set)

for filepath in glob.glob(os.path.join(cqs_directory, '*.txt')):
    label = os.path.splitext(os.path.basename(filepath))[0] 
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            question = line.strip()
            if question:
                question_to_expected_labels[question].add(label.lower())

print(f"Loaded questions from the '{cqs_directory}' directory.")
print(f"Total unique questions found: {len(question_to_expected_labels)}\n")

# --- 2. Iterate through predictions and tally matches ---
matches = 0
non_matches = 0
not_found_in_cqs = 0 

# List to store the details of the failures if we want to print them later
mismatch_records = []

try:
    with open(local_file_name, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1): # Start counting lines at 1
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
                
                if "candidates" in data.get("response", {}):
                    llm_response = data["response"]["candidates"][0]["content"]["parts"][0]["text"].strip().lower()
                    request_key = data.get("key", "")
                    
                    # --- NEW REGEX: Extract the prompt type AND the question ---
                    # Looks for: hyphen + (anything but a hyphen) + '-temp' + (numbers/decimals) + hyphen + (everything else)
                    match = re.search(r'-([^-]+)-temp[\d\.]+-(.*)', request_key)
                    
                    if match:
                        prompt_type = match.group(1).strip() # e.g., 0_shot, cot
                        question = match.group(2).strip()    # e.g., What are...
                        
                        # Extract the representation (nen, axiom, or ttl)
                        key_parts = request_key.split('-')
                        representation = key_parts[1] if len(key_parts) > 1 else "Unknown"

                        valid_labels = question_to_expected_labels.get(question)
                        
                        if valid_labels:
                            if llm_response in valid_labels:
                                matches += 1
                            else:
                                non_matches += 1
                                # Save the details of the failure
                                mismatch_records.append({
                                    "line": line_num,
                                    "representation": representation,
                                    "prompt_type": prompt_type, # <-- Added prompt type
                                    "question": question,
                                    "expected": valid_labels,
                                    "got": llm_response
                                })
                        else:
                            not_found_in_cqs += 1
                    else:
                        print(f"Warning: Could not extract question/prompt_type from key on line {line_num}: {request_key}")
                        
            except json.JSONDecodeError:
                print(f"Error parsing line {line_num}.")

    # --- 3. Print Final Results ---
    print("-" * 40)
    print("RESULTS:")
    print(f"Total Matches (Correct):   {matches}")
    print(f"Total Non-Matches (Wrong): {non_matches}")
    
    total_valid = matches + non_matches
    if total_valid > 0:
        accuracy = (matches / total_valid) * 100
        print(f"Accuracy:                  {accuracy:.2f}%")
        
    if not_found_in_cqs > 0:
        print(f"\nWarning: {not_found_in_cqs} questions from the JSONL were not found in the {cqs_directory} files.")

    # --- 4. Optional: Print Mismatch Details ---
    if SHOW_NON_MATCH_DETAILS and mismatch_records:
        print("\n" + "=" * 60)
        print("🔍 NON-MATCH DETAILS")
        print("=" * 60)
        for record in mismatch_records:
            print(f"Line {record['line']}:")
            print(f"  Format:      {record['representation'].upper()}")
            print(f"  Prompt Type: {record['prompt_type']}") # <-- Displayed here
            print(f"  Question:    {record['question']}")
            print(f"  Expected:    {' OR '.join(record['expected'])}")
            print(f"  Got:         {record['got']}")
            print("-" * 60)

except FileNotFoundError:
    print(f"Error: The file '{local_file_name}' or directory '{cqs_directory}' was not found.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")